# IMEN266 · Ch.8 · Class Problems 13 — Jackson networks: traffic equations, product form, tandem, feedback, routing

**How this notebook works** — every problem follows the same 5 steps:

> **Problem → Model → Analytic → Simulate & Visualize → Experiment**

The analytic answer and the simulation must agree. *That agreement — not the
instructor, not an AI — is your ground truth.*

**In-class version only:** cells marked `# TODO` are for you to complete during
class. A `check(...)` cell right after tells you immediately whether your answer
matches the reference value.

▶ Open in Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch08/notebooks/CP13_jackson_networks.ipynb`

**The facts of this notebook** *(slides pp. 47–66/66; Companion Proof 8)*

- **Traffic equations:** $\lambda_i=\theta_i+\sum_j\lambda_jp_{ji}$, i.e.
  $\boldsymbol\lambda=\boldsymbol\theta(I-P)^{-1}$ — the *effective* arrival rate of a node counts
  every visit, external and internal.
- **Product form (Jackson):** in steady state node $i$ behaves as an $M/M/s_i$ queue with
  arrival rate $\lambda_i$, and the node counts are independent:
  $p(x_1,\ldots,x_N)=\prod_ip_i(x_i)$. The theorem does *not* need the internal flows to be
  Poisson (feedback), but it does need exponential service at every finite-server node.
- **Two Little's laws:** $W_i=L_i/\lambda_i$ is the time per *visit*;
  $W=\sum_iL_i/\sum_i\theta_i$ is the time per *customer*. Visit ratio $v_i=\lambda_i/\sum_k\theta_k$,
  $W=\sum_iv_iW_i$.
- **Design questions answered with the product form:** split or pool servers (pooling wins),
  how to split capacity (square-root rule), and why a new fast link can slow everyone down
  (Braess' paradox: selfish routing $\ne$ social optimum).

In code (`imen266.queueing`): `jackson_solve(theta, P, mu, s)`, `sim_network(theta, P, mu, s, T, rng,
service=, record=)`, `pooling_compare`, `optimal_split`, `braess`, plus `mm1`, `mms`, `mminf`.

In [ ]:
# --- IMEN266 setup (Ch.8): course package + helpers ---------------------------
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:                                   # local clone / PLMS zip: package is on the path
    from imen266.queueing import mm1
except ImportError:                    # Colab: fetch the course package from GitHub (~5 s)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.queueing import mm1
from imen266.queueing import (mms, mminf, mm1k, mmsk, mmkk, erlang_b, erlang_c,
                              birth_death_stationary, truncate, mm1_sojourn_tail, mm1_required_rate,
                              mg1_pk, bulk_mm1, gg1_approx, ggm_approx, reneging_balking_rates,
                              jackson_solve, optimal_split, pooling_compare, braess,
                              SimQueue, simulate_queue, sim_network, little_check, pasta_demo)
from imen266.ctmc import CTMC, birth_death, bd_stationary, poisson_process
from imen266.check import check        # [TODO]/[OK]/[X] self-check (unfinished TODOs never crash)

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11,
                     "legend.frameon": False})

def plot_queue_path(res, T_show=None, ax=None, title=None):
    """Step plot of the number in the system N(t) from a SimQueue / simulate_queue result."""
    ax = ax or plt.gca()
    t, n = res["t"], res["n"]
    if T_show is not None:
        m = t <= T_show
        t, n = np.append(t[m], T_show), np.append(n[m], n[m][-1])
    ax.step(t, n, where="post", lw=1)
    ax.set_xlabel("t"); ax.set_ylabel("number in system N(t)")
    if title: ax.set_title(title)
    return ax

def bd_diagram(births, deaths, states=None, ax=None):
    """Rate diagram of a finite birth-death chain (reuses CTMC.plot_rate_diagram, line layout)."""
    chain = birth_death(births, deaths, states=states)
    return chain.plot_rate_diagram(ax=ax, layout="line")

def ci95(x):
    """Half-width of a 95% confidence interval for the mean of the replications x."""
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / math.sqrt(len(x)) if len(x) > 1 else np.nan

try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    HAS_W = True
except Exception:
    HAS_W = False

def show(fn, **sliders):
    """interact() if widgets are available; otherwise draw once with defaults."""
    if HAS_W:
        interact(fn, **sliders)
    else:
        fn(**{k: getattr(v, "value", v) for k, v in sliders.items()})

print("Setup OK.  imen266.queueing imported.  Widgets available:", HAS_W)

---
## Problem 1 — A three-station network with feedback *(slides pp. 48–57/66)*

> Customers arrive at station 1 as $PP(2)$. After station 1 everybody goes to station 2;
> after station 2 a customer leaves with probability $0.7$ or goes to station 3 with
> probability $0.3$; after station 3 everybody returns to station 1. All stations have a
> single exponential server, with rates $\mu=(4,3,2)$. How many customers are in the
> network on average, and how long does a customer spend in it?

### Model
An open Jackson network with $\boldsymbol\theta=(2,0,0)$ and routing matrix
$P=\begin{pmatrix}0&1&0\\0&0&0.3\\1&0&0\end{pmatrix}$ (row $i$ = where a customer goes *after*
station $i$; the missing mass of a row is the exit probability).
Step 1: traffic equations $\lambda_1=2+\lambda_3$, $\lambda_2=\lambda_1$, $\lambda_3=0.3\lambda_2$
(slide p. 56/66). Step 2: each node is $M/M/1$ with its own $\lambda_i$ (slide p. 57/66).

In [ ]:
theta = np.array([2.0, 0.0, 0.0])
mu = np.array([4.0, 3.0, 2.0])

# TODO 1: the routing matrix (row i: probabilities of going to 1, 2, 3 after station i), then solve
#         the traffic equations lam = theta + lam P for lam, and the utilisations rho_i = lam_i / mu_i
P = ...
lam = ...
rho = ...
print("lam =", lam, "  rho =", rho)

In [ ]:
ref = jackson_solve(theta, [[0, 1, 0], [0, 0, 0.3], [1, 0, 0]], mu)
check("lam_1 = 2/0.7",                lam if lam is Ellipsis else lam[0], 2/0.7, tol=1e-6)
check("lam_3 = 0.3 lam_2",            lam if lam is Ellipsis else lam[2], 0.6/0.7, tol=1e-6)
check("max |lam - jackson_solve|",    None if lam is Ellipsis else float(np.abs(np.asarray(lam) - ref["lam"]).max()), 0.0, tol=1e-6)
check("rho_2 (the bottleneck)",       rho if rho is Ellipsis else rho[1], 2/0.7/3, tol=1e-6)

### Analytic: node by node, then the whole network
$L_i=\lambda_i/(\mu_i-\lambda_i)$ (each node is $M/M/1$), $W_i=L_i/\lambda_i$ per visit,
$L=\sum_iL_i$, $W=L/\sum_i\theta_i$ per customer; visit ratios $v_i=\lambda_i/\sum_k\theta_k$
and $W=\sum_iv_iW_i$ (slide p. 55/66).

In [ ]:
# TODO 2: L_i of each M/M/1 node (with ITS effective rate), W_i per visit, the network L and W, and the visit ratios v_i
L_i = ...
W_i = ...
L = ...
W = ...
v = ...
print("L_i =", L_i, " W_i =", W_i, " L =", L, " W =", W, " v =", v)

In [ ]:
check("L_2 (bottleneck)",             L_i if L_i is Ellipsis else L_i[1], 20.0,   tol=1e-6)
check("W_2 per visit",                W_i if W_i is Ellipsis else W_i[1], 7.0,    tol=1e-6)
check("network L",                    L, 23.25,  tol=1e-6)
check("network W per customer",       W, 11.625, tol=1e-6)
check("visits to station 3",          v if v is Ellipsis else v[2], 0.3/0.7, tol=1e-6)

In [ ]:
# --- Simulate the network and compare (reference numbers from jackson_solve, independent of your TODOs) -------------
ref = jackson_solve(theta, [[0, 1, 0], [0, 0, 0.3], [1, 0, 0]], mu)
sim = sim_network(theta, [[0, 1, 0], [0, 0, 0.3], [1, 0, 0]], mu, T=80_000, rng=rng, warmup=2_000)
print(f"{'':>10} {'L_1':>7} {'L_2':>7} {'L_3':>7} {'L':>7} {'W':>7}   {'W_1':>6} {'W_2':>6} {'W_3':>6} (per visit)")
print(f"{'theory':>10} {ref['L_i'][0]:>7.3f} {ref['L_i'][1]:>7.3f} {ref['L_i'][2]:>7.3f} {ref['L']:>7.3f} {ref['W']:>7.3f}   {ref['W_i'][0]:>6.3f} {ref['W_i'][1]:>6.3f} {ref['W_i'][2]:>6.3f}")
print(f"{'simulation':>10} {sim['L_i'][0]:>7.3f} {sim['L_i'][1]:>7.3f} {sim['L_i'][2]:>7.3f} {sim['L']:>7.3f} {sim['W']:>7.3f}   {sim['W_i'][0]:>6.3f} {sim['W_i'][1]:>6.3f} {sim['W_i'][2]:>6.3f}")
print("Station 2 runs at rho = 0.95: its queue has long excursions, so even 80,000 time units leave a few percent of noise there.")
plt.figure(figsize=(7, 3.6))
x = np.arange(3); plt.bar(x - 0.2, ref["L_i"], 0.4, label="product form"); plt.bar(x + 0.2, sim["L_i"], 0.4, label="simulation")
plt.xticks(x, ["station 1", "station 2", "station 3"]); plt.ylabel("mean number L_i"); plt.legend()
plt.title("Where are the customers? Almost all of them at the bottleneck"); plt.show()

In [ ]:
def bottleneck_demo(mu2=3.0):
    """Mean number at each station and the sojourn W as the rate of station 2 varies."""
    m2 = np.linspace(2.9, 6.0, 200)
    Ws = [jackson_solve(theta, [[0, 1, 0], [0, 0, 0.3], [1, 0, 0]], [4.0, m, 2.0])["W"] for m in m2]
    j = jackson_solve(theta, [[0, 1, 0], [0, 0, 0.3], [1, 0, 0]], [4.0, mu2, 2.0])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].bar(["1", "2", "3"], j["L_i"]); ax[0].set_ylabel("L_i"); ax[0].set_title(f"mu_2 = {mu2}: L = {j['L']:.2f}, W = {j['W']:.2f}")
    ax[1].plot(m2, Ws); ax[1].axvline(mu2, color="r", alpha=.5); ax[1].axvline(2/0.7, color="k", ls=":")
    ax[1].set_xlabel("mu_2"); ax[1].set_ylabel("W per customer"); ax[1].set_ylim(0, 40); ax[1].set_title("W explodes as mu_2 -> lam_2 = 2.857")
    plt.show()

show(bottleneck_demo, mu2=FloatSlider(min=2.9, max=6.0, step=0.1, value=3.0) if HAS_W else 3.0)

**Think about it** — (i) $\lambda_1=2.857>\theta_1=2$: where do the extra arrivals come from, and
why is the *network* throughput still $2$? (ii) A customer visits station 1 on average
$1/0.7$ times; derive this from the geometric number of loops. (iii) The symbolic example of
slides pp. 50–51/66 has three external flows and six servers at one node — write its routing
matrix and solve it with `jackson_solve` for $\theta=(1,0.5,0.5)$, $p_1=0.4,p_2=0.6,p_3=0.5,p_4=0.5$,
$\mu=(1,2,0.5)$, $s=(3,1,6)$.

---
## Problem 2 — Bay-Gull Bagels *(slides pp. 58–60/66; the course handout)*

> Bay-Gull Bagels is a bagel store in downtown Pohang. Customers arrive at the **Bagels**
> counter ($PP(1)$ per minute), at the **Smoothies** counter ($PP(0.2)$) and at the
> self-service **Drinks & Coffee** station ($PP(0.4)$). After Bagels, 20 % go to Smoothies,
> 30 % to Drinks and 50 % straight to the **Cashier**; everybody from Smoothies and Drinks
> goes to the Cashier; after paying, half the customers leave and half **Eat in**; of those
> who eat in, 5 % go back for another drink and the rest leave. Service: Bagels 3 servers,
> mean 2.5 min; Smoothies 2 servers, 4.5 min; Drinks self-service, 2 min; Cashier 2 servers,
> 1 min; Eat in self-service, 20 min. All service times are exponential. How many
> customers are at each station and in the whole store, and how long does a visit take?

### Model
Five stations, $\boldsymbol\theta=(1,0.2,0.4,0,0)$ per minute, $s=(3,2,\infty,2,\infty)$,
$1/\mu=(2.5,4.5,2,1,20)$ min, routing read off the picture (slide p. 59/66):
$p_{12}=0.2,\ p_{13}=0.3,\ p_{14}=0.5,\ p_{24}=p_{34}=1,\ p_{45}=0.5,\ p_{53}=0.05$.
Step 1 traffic equations, Step 2 stability ($\rho_i=\lambda_i/(s_i\mu_i)<1$ at the
finite-server stations), Step 3 five independent $M/M/s$ queues (slide p. 60/66).

In [ ]:
theta_b = np.array([1.0, 0.2, 0.4, 0.0, 0.0])                  # per minute
mean_s  = np.array([2.5, 4.5, 2.0, 1.0, 20.0]); mu_b = 1 / mean_s
s_b     = [3, 2, np.inf, 2, np.inf]
names = ["Bagels", "Smoothies", "Drinks", "Cashier", "Eat in"]

# TODO 3: the 5x5 routing matrix (station order: Bagels, Smoothies, Drinks, Cashier, Eat in), the effective
#         arrival rates lam_b from the traffic equations, and the utilisations rho_b = lam_i / (s_i mu_i)
#         (use 0 for the self-service stations)
P_b = ...
lam_b = ...
rho_b = ...
print("lam =", lam_b, "\nrho =", rho_b)

In [ ]:
check("lam_3 (Drinks) = 0.741",        None if lam_b is Ellipsis else lam_b[2], 0.7225/0.975, tol=1e-6)
check("lam_4 (Cashier) = 1.641",       None if lam_b is Ellipsis else lam_b[3], 0.9 + 0.7225/0.975, tol=1e-6)
check("lam_5 (Eat in) = 0.821",        None if lam_b is Ellipsis else lam_b[4], 0.5*(0.9 + 0.7225/0.975), tol=1e-6)
check("rho_2 (Smoothies) = 0.9",       None if rho_b is Ellipsis else rho_b[1], 0.9, tol=1e-6)
check("rows of P_b sum to at most 1",  None if P_b is Ellipsis else float(np.asarray(P_b).sum(axis=1).max()), 1.0, tol=1e-9)

### Analytic: five independent queues
$M/M/\infty$: $L_i=\lambda_i/\mu_i$. $M/M/s$: $L_i=L_{q,i}+\lambda_i/\mu_i$ with the $L_q$ of
slide p. 19/66 — use `mms(lam, mu, s)["L"]` and `mminf(lam, mu)["L"]`. Then $L=\sum_iL_i$ and
$W=L/\sum_i\theta_i=L/1.6$ min per customer.

In [ ]:
# TODO 4: L_i of each station (mms for s finite, mminf for self-service), the store total L and W per customer
L_b = ...
L_store = ...
W_store = ...
print("L_i =", L_b, " L =", L_store, " W =", W_store, "min")

In [ ]:
ref_b = jackson_solve(theta_b, [[0, .2, .3, .5, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, .5], [0, 0, .05, 0, 0]], mu_b, s_b)
check("L (Smoothies) = 9.47",          None if L_b is Ellipsis else L_b[1], ref_b["L_i"][1], tol=1e-3)
check("L (Eat in) = 16.41",            None if L_b is Ellipsis else L_b[4], ref_b["L_i"][4], tol=1e-3)
check("store L = 38.4",                L_store, ref_b["L"], tol=1e-3)
check("store W = 24.0 min",            W_store, ref_b["W"], tol=1e-3)

In [ ]:
# --- Simulate the store: 4 independent runs of 50,000 minutes each (about 35 days of round-the-clock trade) ---------
P_ref = [[0, .2, .3, .5, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, .5], [0, 0, .05, 0, 0]]
runs = [sim_network(theta_b, P_ref, mu_b, s_b, T=50_000, rng=rng, warmup=1_000) for _ in range(4)]
L_runs = np.array([r["L_i"] for r in runs]); Wv_runs = np.array([r["W_i"] for r in runs]); W_runs = np.array([r["W"] for r in runs])
print(f"{'station':>10} {'L theory':>9} {'L sim (95% CI)':>18}   {'W/visit theory':>15} {'W/visit sim':>12}")
for i, nm in enumerate(names):
    print(f"{nm:>10} {ref_b['L_i'][i]:>9.2f} {L_runs[:, i].mean():>9.2f} +- {ci95(L_runs[:, i]):<5.2f}   {ref_b['W_i'][i]:>15.2f} {Wv_runs[:, i].mean():>12.2f}")
print(f"{'store':>10} {ref_b['L']:>9.2f} {L_runs.sum(1).mean():>9.2f} +- {ci95(L_runs.sum(1)):<5.2f}   W per customer: theory {ref_b['W']:.2f} min, simulation {W_runs.mean():.2f} +- {ci95(W_runs):.2f} min")
print("Smoothies runs at rho = 0.9 with only 0.4 customers a minute: its queue drifts for hours at a time, hence the widest error bar.")

In [ ]:
# --- What breaks: make the cashier's service time deterministic (1 min exactly) --------------------------------
det_cashier = [None, None, None, lambda r, n: np.full(n, 1.0), None]
runs_d = [sim_network(theta_b, P_ref, mu_b, s_b, T=50_000, rng=rng, warmup=1_000, service=det_cashier) for _ in range(4)]
L_det = np.array([r["L_i"] for r in runs_d])
print(f"{'station':>10} {'product form':>13} {'exp. service':>13} {'det. cashier':>13}")
for i, nm in enumerate(names):
    print(f"{nm:>10} {ref_b['L_i'][i]:>13.2f} {L_runs[:, i].mean():>13.2f} {L_det[:, i].mean():>13.2f}")
print(f"{'store':>10} {ref_b['L']:>13.2f} {L_runs.sum(1).mean():>13.2f} {L_det.sum(1).mean():>13.2f}")
print("The cashier is now an M/D/2 station: a shorter queue than the M/M/2 formula says (Pollaczek-Khinchine logic),")
print("so the product form overestimates L_4. The other stations do not move beyond their error bars: upstream of the")
print("cashier nothing changed, and downstream only self-service stations follow, whose L = lam/mu is insensitive.")
plt.figure(figsize=(8, 3.6))
x = np.arange(5)
plt.bar(x - 0.27, ref_b["L_i"], 0.27, label="product form")
plt.bar(x, L_runs.mean(0), 0.27, yerr=[ci95(L_runs[:, i]) for i in range(5)], capsize=3, label="simulation, exponential")
plt.bar(x + 0.27, L_det.mean(0), 0.27, yerr=[ci95(L_det[:, i]) for i in range(5)], capsize=3, label="simulation, deterministic cashier")
plt.xticks(x, names); plt.ylabel("mean number L_i"); plt.legend(); plt.title("Bay-Gull Bagels: where the 38 customers are"); plt.show()

In [ ]:
def bagels_demo(rush=1.0, s_smoothie=2, s_cashier=2):
    """Scale all external arrival rates by `rush` and change the staffing of Smoothies / Cashier."""
    s_new = [3, s_smoothie, np.inf, s_cashier, np.inf]
    j = jackson_solve(rush * theta_b, P_ref, mu_b, s_new)
    rho = j["rho"]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    if not j["stable"]:
        bad = [names[i] for i in range(5) if rho[i] >= 1]
        ax.text(0.5, 0.5, f"unstable: {', '.join(bad)} (rho >= 1)", ha="center", va="center", transform=ax.transAxes, fontsize=14, color="r")
        ax.set_axis_off(); plt.show(); return
    ax.bar(names, j["L_i"]); ax.set_ylabel("L_i")
    ax.set_title(f"rush x{rush:.2f}, Smoothies {s_smoothie} srv, Cashier {s_cashier} srv:  L = {j['L']:.1f},  W = {j['W']:.1f} min,  max rho = {rho.max():.2f}")
    plt.show()

show(bagels_demo, rush=FloatSlider(min=0.5, max=1.3, step=0.05, value=1.0) if HAS_W else 1.0,
     s_smoothie=IntSlider(min=1, max=4, value=2) if HAS_W else 2,
     s_cashier=IntSlider(min=1, max=4, value=2) if HAS_W else 2)

**Think about it** — (i) Smoothies carries $0.4$ customers per minute, Bagels $1.0$ — yet
Smoothies holds more people. Which single number explains it? (ii) By how much can the
lunch rush grow (all $\theta_i$ scaled by the same factor) before the store becomes
unstable, and which station fails first? (iii) Eat-in holds 16 of the 38 customers but
nobody waits there: is $L$ or $L_q$ the number the manager should watch at each station?
(iv) Why did making the cashier deterministic leave Eat-in unchanged although Eat-in
receives the cashier's (no longer Poisson) output?

---
## Problem 3 — Tandem queue: Burke's theorem and independence *(slides pp. 61–62/66)*

> Jobs arrive as $PP(2)$ at a two-stage line: stage 1 is $\exp(4)$, stage 2 is $\exp(3)$,
> single servers, infinite buffers, FCFS. Find $L$ and $W$, and check three claims of the
> theory by simulation: the output of stage 1 is $PP(2)$ (Burke); $X_1(t)$ and $X_2(t)$ are
> independent at any fixed $t$ (product form); the two sojourn times of the same job are
> independent (Reich).

### Model
$\lambda_1=\lambda_2=\lambda=2$ (what goes in must come out); $\rho=(0.5,\ 2/3)$;
$p(x_1,x_2)=(1-\rho_1)\rho_1^{x_1}(1-\rho_2)\rho_2^{x_2}$; $L=\sum_i\lambda/(\mu_i-\lambda)$,
$W=\sum_i1/(\mu_i-\lambda)$ (slide p. 62/66).

In [ ]:
lam_t, mu_t = 2.0, np.array([4.0, 3.0])

# TODO 5: utilisations, L and W of the tandem line, and the product-form probability that both stages are empty
rho_t = ...
L_t = ...
W_t = ...
p00 = ...
print("rho =", rho_t, " L =", L_t, " W =", W_t, " P(0,0) =", p00)

In [ ]:
check("L of the tandem",            L_t, 3.0,  tol=1e-6)
check("W of the tandem",            W_t, 1.5,  tol=1e-6)
check("P(X1 = 0, X2 = 0) = 1/6",    p00, 1/6,  tol=1e-6)

In [ ]:
# --- Simulate with record=True and test the three claims --------------------------------------------------------
T_t, warm = 20_000, 500
sim_t = sim_network([lam_t, 0], [[0, 1], [0, 0]], [4.0, 3.0], T=T_t, rng=rng, warmup=warm, record=True)
print(f"L sim {sim_t['L']:.3f} (theory 3),  W sim {sim_t['W']:.3f} (theory 1.5),  per stage W_i {sim_t['W_i'].round(3)} (theory 0.5, 1.0)")

# (a) Burke: inter-departure times of stage 1 look exponential with rate 2 and are uncorrelated
dep1 = sim_t["departures_from"][0]; gaps = np.diff(dep1[dep1 > warm])
print(f"(a) Burke:  mean gap {gaps.mean():.4f} (1/lam = 0.5)   cv {gaps.std()/gaps.mean():.3f} (1 for exponential)   lag-1 autocorrelation {np.corrcoef(gaps[:-1], gaps[1:])[0, 1]:+.3f}")

# (b) product form at a fixed time: joint time-average distribution vs product of the marginals
t_path, n_path = sim_t["t"], sim_t["n_path"]
grid = np.arange(warm, T_t, 0.01)                                    # sample the state every 0.01 time units
X = n_path[np.searchsorted(t_path, grid, side="right") - 1]
Kmax = 10
joint = np.zeros((Kmax, Kmax))
np.add.at(joint, (np.minimum(X[:, 0], Kmax - 1), np.minimum(X[:, 1], Kmax - 1)), 1); joint /= joint.sum()
tv = 0.5 * np.abs(joint - np.outer(joint.sum(1), joint.sum(0))).sum()
print(f"(b) independence at a fixed t:  TV distance between the joint law and the product of marginals = {tv:.3f};  corr(X1(t), X2(t)) = {np.corrcoef(X[:, 0], X[:, 1])[0, 1]:+.3f}")
print("    ... but the PROCESS is not independent: corr(X1(t), X2(t + d)) for d = ", end="")
print(",  ".join(f"{d}: {np.corrcoef(X[:len(X) - int(d/0.01), 0], X[int(d/0.01):, 1])[0, 1]:+.3f}" for d in (0.25, 0.5, 1.0, 2.0)))

# (c) Reich: FCFS single servers keep the order, so the k-th arrival is the k-th departure at both stages
a1, d1 = sim_t["arrivals_at"][0], sim_t["departures_from"][0]
a2, d2 = sim_t["arrivals_at"][1], sim_t["departures_from"][1]
k = min(len(d1), len(d2)); s1, s2 = d1[:k] - a1[:k], d2[:k] - a2[:k]
print(f"(c) Reich:  mean sojourns {s1.mean():.3f}, {s2.mean():.3f} (0.5, 1.0);  corr(sojourn at 1, sojourn at 2) = {np.corrcoef(s1, s2)[0, 1]:+.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(gaps, bins=60, density=True, alpha=.6, label="inter-departure times, stage 1")
xx = np.linspace(0, gaps.max(), 200); ax[0].plot(xx, lam_t * np.exp(-lam_t * xx), "k", label="exp(2) density")
ax[0].set_xlim(0, 3); ax[0].set_xlabel("gap"); ax[0].legend(); ax[0].set_title("Burke: the output of an M/M/1 is Poisson", fontsize=10)
lags = np.arange(0, 3.01, 0.1)
cors = [np.corrcoef(X[:len(X) - int(round(d/0.01)), 0], X[int(round(d/0.01)):, 1])[0, 1] for d in lags]
ax[1].plot(lags, cors, "o-"); ax[1].axhline(0, color="k", lw=.8); ax[1].set_xlabel("lag d"); ax[1].set_ylabel("corr(X1(t), X2(t + d))")
ax[1].set_title("Independent at the same instant, not across time", fontsize=10); plt.show()

In [ ]:
def tandem_demo(mu1=4.0, mu2=3.0):
    """Product-form joint pmf of (X1, X2) for PP(2) input and the resulting L, W."""
    lam = 2.0
    if min(mu1, mu2) <= lam:
        print("unstable: both rates must exceed lam = 2"); return
    r1, r2 = lam / mu1, lam / mu2
    x = np.arange(12)
    pm = np.outer((1 - r1) * r1**x, (1 - r2) * r2**x)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    im = ax.imshow(pm.T, origin="lower", cmap="Blues"); plt.colorbar(im, ax=ax, label="p(x1, x2)")
    ax.set_xlabel("x1 (stage 1)"); ax.set_ylabel("x2 (stage 2)"); ax.grid(False)
    ax.set_title(f"L = {lam/(mu1-lam) + lam/(mu2-lam):.2f},  W = {1/(mu1-lam) + 1/(mu2-lam):.2f}:  rows and columns are proportional")
    plt.show()

show(tandem_demo, mu1=FloatSlider(min=2.2, max=8, step=0.1, value=4.0) if HAS_W else 4.0,
     mu2=FloatSlider(min=2.2, max=8, step=0.1, value=3.0) if HAS_W else 3.0)

**Think about it** — (i) Why does the lag-$d$ correlation start at $0$ and *grow* with
$d$? (What does a large $X_1(t)$ announce for stage 2 a little later?) (ii) Swap the two
rates ($\mu=(3,4)$): $L$, $W$ and the joint law are unchanged — explain with the product
form; is the *sample path* unchanged too? (iii) Burke's theorem needs steady state and
exponential service: what happens to the stage-1 output if stage 1 is $M/D/1$? (Try
`service=[lambda r, n: np.full(n, 0.25), None]`.)

---
## Problem 4 — Feedback queue: one station, two Little's laws *(slides pp. 63–64/66)*

> A single $\exp(\mu)$ server, $\mu=4$, receives $PP(\lambda)$, $\lambda=1$. After each service the
> customer leaves with probability $1-q$ or rejoins the end of the queue with probability
> $q=0.5$ (rework). How many customers are in the system, how long does a customer stay,
> and how long does one *visit* take?

### Model
**CTMC view:** birth rate $\lambda$, death rate $\mu(1-q)$ (a fed-back completion does not
change the count) — an $M/M/1$ with service rate $\mu(1-q)=2$: $L=\lambda/(\mu(1-q)-\lambda)$,
$W_{\text{customer}}=1/(\mu(1-q)-\lambda)$. **Network view:** one node with $p_{11}=q$,
$\hat\lambda=\lambda/(1-q)$ (external + fed back), $W_{\text{node}}=L/\hat\lambda$ per visit; a
customer makes $1/(1-q)$ visits, so $W_{\text{customer}}=W_{\text{node}}/(1-q)$ (slide p. 64/66).

In [ ]:
lam_f, mu_f, q = 1.0, 4.0, 0.5

# TODO 6: L and W per customer (CTMC view), the effective rate lam_hat and W per visit (network view), mean visits
L_f = ...
W_cust = ...
lam_hat = ...
W_node = ...
visits = ...
print("L =", L_f, " W_customer =", W_cust, " lam_hat =", lam_hat, " W_node =", W_node, " visits =", visits)

In [ ]:
check("L",                      L_f,     1.0,  tol=1e-6)
check("W per customer",         W_cust,  1.0,  tol=1e-6)
check("effective rate lam_hat", lam_hat, 2.0,  tol=1e-6)
check("W per visit",            W_node,  0.5,  tol=1e-6)
check("visits per customer",    visits,  2.0,  tol=1e-6)

In [ ]:
# --- Simulate: W per customer vs W per visit, a non-Poisson input, and still a geometric X ---------------------------
sim_f = sim_network([lam_f], [[q]], [mu_f], T=40_000, rng=rng, warmup=500, record=True)
print(f"simulation: L = {sim_f['L']:.3f}   W per customer = {sim_f['W']:.3f}   W per visit = {sim_f['W_i'][0]:.3f}   (theory 1, 1, 0.5)")
gaps_f = np.diff(sim_f["arrivals_at"][0]); gaps_f = gaps_f[gaps_f > 0]
print(f"input of the node (external + fed back): rate {1/gaps_f.mean():.3f} (lam_hat = 2),  cv of the gaps {gaps_f.std()/gaps_f.mean():.3f}  (a Poisson process would have cv = 1)")
t_f, n_f = sim_f["t"], sim_f["n_path"][:, 0]
grid = np.arange(500, 40_000, 0.01); Xf = n_f[np.searchsorted(t_f, grid, side="right") - 1]
p_sim = np.bincount(Xf, minlength=8)[:8] / len(Xf)
rho_f = lam_f / (mu_f * (1 - q)); p_geo = (1 - rho_f) * rho_f ** np.arange(8)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(gaps_f, bins=80, density=True, alpha=.6, label="gaps between arrivals at the node")
xx = np.linspace(0, 3, 200); ax[0].plot(xx, 2 * np.exp(-2 * xx), "k", label="exp(2): Poisson of the same rate")
ax[0].set_xlim(0, 2.5); ax[0].legend(); ax[0].set_title("Node input: not Poisson (fed-back customers return at once)", fontsize=10)
ax[1].bar(np.arange(8), p_sim, alpha=.6, label="time average (simulation)"); ax[1].plot(np.arange(8), p_geo, "ko-", label="geometric, rho = lam / (mu(1-q))")
ax[1].set_xlabel("customers in the system"); ax[1].legend(); ax[1].set_title("... yet the number in system is M/M/1-like", fontsize=10)
plt.show()

In [ ]:
def feedback_demo(q=0.5):
    """W per customer and per visit as the feedback probability varies (lam = 1, mu = 4)."""
    qq = np.linspace(0, 0.74, 200)
    Wc = 1 / (4 * (1 - qq) - 1); Wn = (1 - qq) / (4 * (1 - qq) - 1)
    plt.figure(figsize=(7, 3.8)); plt.plot(qq, Wc, label="W per customer"); plt.plot(qq, Wn, label="W per visit")
    plt.axvline(q, color="r", alpha=.5); plt.ylim(0, 8); plt.xlabel("feedback probability q"); plt.legend()
    plt.title(f"q = {q}: W_customer = {1/(4*(1-q)-1):.2f}, W_visit = {(1-q)/(4*(1-q)-1):.2f}, visits = {1/(1-q):.2f}"); plt.show()

show(feedback_demo, q=FloatSlider(min=0, max=0.7, step=0.05, value=0.5) if HAS_W else 0.5)

**Think about it** — (i) The stability condition is $\lambda<\mu(1-q)$, not $\lambda<\mu$: give
the one-sentence reason. (ii) Both Little's laws are correct — the difference is only the
arrival rate they use. Which $W$ would a customer complain about, and which one would the
server's log show? (iii) The product form held although the input was not Poisson. Which
step of the Companion proof (Proof 8(b)) never mentions Poisson inputs at all?

---
## Problem 5 — Final-style: split or pool? *(slide p. 54/66; 2025 final Q4, 2024 final Q1)*

> **(a)** Two customer streams, each $PP(\lambda)$ with $\lambda=1$, are served by two
> $\exp(\mu)$ servers, $\mu=1.5$. Design A: each stream has its own server and queue.
> Design B: one common queue, both servers serve whoever is first. Compare $L$ and $W$.
>
> **(b)** A $PP(\lambda)$ stream, $\lambda=2$, is split by type: 80 % go to server 1,
> 20 % to server 2. A total service capacity $\mu_1+\mu_2=4$ can be allocated freely.
> Which allocation minimises the mean sojourn $W=\sum_ip_i/(\mu_i-p_i\lambda)$?

### Model
(a) Design A: two independent $M/M/1$ queues (Poisson splitting), each with
$\lambda=1$: $W_A=1/(\mu-\lambda)$. Design B: one $M/M/2$ queue with arrival rate $2\lambda$.
(b) Lagrange on $\sum_ip_i/(\mu_i-p_i\lambda)$ subject to $\sum_i\mu_i=\mu_{\text{tot}}$ gives the
**square-root rule** $\mu_i^*=p_i\lambda+\sqrt{p_i}\,\dfrac{\mu_{\text{tot}}-\lambda}{\sum_j\sqrt{p_j}}$:
the spare capacity $\mu_{\text{tot}}-\lambda$ is shared in proportion to $\sqrt{p_i}$, not to $p_i$.

In [ ]:
lam_p, mu_p = 1.0, 1.5
# TODO 7 (a): W and L of the split design (two M/M/1) and of the pooled design (one M/M/2 with rate 2 lam)
W_A = ...
L_A = ...
W_B = ...
L_B = ...
print("split: L =", L_A, " W =", W_A, "   pooled: L =", L_B, " W =", W_B)

lam_s, p_s, mu_tot = 2.0, np.array([0.8, 0.2]), 4.0
# TODO 8 (b): the square-root allocation mu_star (a length-2 array) and the resulting W_star; compare with the equal
#             split (2, 2) and the proportional split (3.2, 0.8) using W = sum_i p_i / (mu_i - p_i lam)
mu_star = ...
W_star = ...
W_equal = ...
W_prop = ...
print("mu* =", mu_star, " W* =", W_star, " W_equal =", W_equal, " W_prop =", W_prop)

In [ ]:
pc = pooling_compare(1.0, 1.5); osp = optimal_split(2.0, [0.8, 0.2], 4.0)
check("(a) W split",            W_A, pc["W_split"],  tol=1e-6)
check("(a) W pooled",           W_B, pc["W_pooled"], tol=1e-6)
check("(a) L pooled = 0.6 L split", None if (L_A is Ellipsis or L_B is Ellipsis) else L_B / L_A, pc["ratio_L"], tol=1e-6)
check("(b) mu_1*",              None if mu_star is Ellipsis else mu_star[0], osp["mu_opt"][0], tol=1e-6)
check("(b) W*",                 W_star,  osp["W_opt"],   tol=1e-6)
check("(b) W of the equal split", W_equal, osp["W_equal"], tol=1e-6)
check("(b) W of the proportional split", W_prop, 2 / (4.0 - 2.0), tol=1e-6)

In [ ]:
# --- Simulate design A and design B, and draw W as a function of the allocation in (b) ---------------------------
sA = simulate_queue(1.0, 1.5, T=60_000, rng=rng, warmup=1_000)                 # one of the two identical M/M/1 queues
sB = simulate_queue(2.0, 1.5, T=60_000, rng=rng, warmup=1_000, s=2)            # the pooled M/M/2
print(f"(a) simulation: split W = {sA['W']:.3f} (theory {pc['W_split']:.3f}),  pooled W = {sB['W']:.3f} (theory {pc['W_pooled']:.3f})")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].bar(["split (2 x M/M/1)", "pooled (M/M/2)"], [pc["L_split"], pc["L_pooled"]]); ax[0].set_ylabel("L (both streams)")
ax[0].set_title("(a) pooling: 40 % fewer customers", fontsize=10)
m1 = np.linspace(1.62, 3.58, 300); m2 = 4.0 - m1
Wc = 0.8 / (m1 - 1.6) + 0.2 / (m2 - 0.4)
ax[1].plot(m1, Wc); ax[1].set_ylim(0.8, 3); ax[1].set_xlabel("mu_1  (mu_2 = 4 - mu_1)"); ax[1].set_ylabel("W")
for m, lab in [(osp["mu_opt"][0], "square root"), (2.0, "equal"), (3.2, "proportional")]:
    ax[1].scatter([m], [0.8 / (m - 1.6) + 0.2 / (4 - m - 0.4)], zorder=3, s=60, label=f"{lab}: mu_1 = {m:.2f}")
ax[1].legend(); ax[1].set_title("(b) square-root rule vs the naive splits", fontsize=10); plt.show()

In [ ]:
def split_demo(p1=0.8, mu_tot=4.0):
    """Optimal allocation of a total capacity mu_tot to two M/M/1 servers that receive fractions p1, 1 - p1 of PP(2)."""
    lam = 2.0; p = np.array([p1, 1 - p1])
    if mu_tot <= lam:
        print("need mu_tot > lam = 2"); return
    o = optimal_split(lam, p, mu_tot)
    m1 = np.linspace(p1 * lam + 1e-3, mu_tot - (1 - p1) * lam - 1e-3, 400); W = p1 / (m1 - p1 * lam) + (1 - p1) / (mu_tot - m1 - (1 - p1) * lam)
    plt.figure(figsize=(7, 3.8)); plt.plot(m1, W); plt.ylim(0, min(10, np.nanmax(W)))
    plt.axvline(o["mu_opt"][0], color="r", alpha=.6, label=f"square root: mu_1* = {o['mu_opt'][0]:.2f}, W* = {o['W_opt']:.3f}")
    plt.axvline(p1 * mu_tot, color="g", ls=":", label=f"proportional: W = {2/(mu_tot-lam):.3f}")
    plt.xlabel("mu_1"); plt.ylabel("W"); plt.legend(); plt.title(f"p = ({p1:.2f}, {1-p1:.2f}), mu_tot = {mu_tot}"); plt.show()

show(split_demo, p1=FloatSlider(min=0.5, max=0.95, step=0.05, value=0.8) if HAS_W else 0.8,
     mu_tot=FloatSlider(min=2.5, max=8, step=0.25, value=4.0) if HAS_W else 4.0)

**Think about it** — (i) In (a) both designs have the same utilisation $\rho=2/3$ per
server; where does the pooled system gain? (Think of the state "one server idle, one
customer waiting".) (ii) In (b) the proportional split gives every server the same $\rho$
and the same $W_i$ — why is it still not optimal? (iii) What would change in (b) if the two
types had *different* service requirements? (iv) On the 2025 final, part (a) was asked for
$M/M/2$ versus two $M/M/1$ — write the two $L$'s as closed-form expressions in $\rho=\lambda/\mu$.

---
## Problem 6 — Braess' paradox *(slides pp. 65–66/66)*

> Demand $PP(2\lambda)$, $\lambda=1$, can take two routes: queue $x$ followed by a fixed delay
> of $2$, or a fixed delay of $2$ followed by queue $y$. Both queues are $\exp(\mu)$ single
> servers with $\mu=2.5$. A new link of fixed delay $1$ is built from $x$ to $y$, creating the
> route $x\to\text{link}\to y$. Customers choose the route with the smallest mean sojourn
> (Wardrop equilibrium). Does the link help?

### Model
**Before:** by symmetry half the demand takes each route; each queue is $M/M/1(\lambda,\mu)$,
$W_{\text{before}}=1/(\mu-\lambda)+2$. **After:** write $x$, $y$ for the mean sojourns at the two
queues; in equilibrium all used routes take the same time, $x+2=x+1+y=2+y$, so $x=y=1$:
each queue carries flow $\mu-1$, and the route flows follow from
$f_x=f_{\text{up}}+f_{\text{mid}}$, $f_y=f_{\text{mid}}+f_{\text{low}}$, $f_{\text{up}}+f_{\text{mid}}+f_{\text{low}}=2\lambda$.
The paradox: $W_{\text{after}}=3>W_{\text{before}}$ whenever $\mu>\lambda+1$.

In [ ]:
lam_r, mu_r = 1.0, 2.5

# TODO 9: W before the link; after the link the flow through each queue (from x = y = 1), the middle-route flow,
#         the upper (= lower) route flow, and W after the link
W_before = ...
f_queue = ...
f_mid = ...
f_up = ...
W_after = ...
print("W_before =", W_before, " f_queue =", f_queue, " f_mid =", f_mid, " f_up =", f_up, " W_after =", W_after)

In [ ]:
br = braess(1.0, 2.5)
check("W before the link",           W_before, br["W_before"], tol=1e-6)
check("flow through each queue",     f_queue,  2.5 - 1,        tol=1e-6)
check("flow on the new middle route", f_mid,   br["flows"][1], tol=1e-6)
check("flow on the upper route",     f_up,     br["flows"][0], tol=1e-6)
check("W after the link",            W_after,  br["W_after"],  tol=1e-6)

In [ ]:
# --- Simulate both road networks: the fixed delays are self-service stations with deterministic service -----------
delay2 = lambda r, n: np.full(n, 2.0); delay1 = lambda r, n: np.full(n, 1.0)
# before: stations [x, delay after x, delay before y, y]; half of PP(2) enters at x, half at the lower delay
P_before = np.zeros((4, 4)); P_before[0, 1] = 1; P_before[2, 3] = 1
s_before = sim_network([1.0, 0, 1.0, 0], P_before, [2.5, 1, 1, 2.5], [1, np.inf, np.inf, 1], T=40_000, rng=rng, warmup=1_000,
                       service=[None, delay2, delay2, None])
# after: stations [x, delay after x, delay before y, y, link]; equilibrium flows: 1.5 enter at x (1/3 keep the upper route,
# 2/3 take the link), 0.5 enter at the lower delay
P_after = np.zeros((5, 5)); P_after[0, 1] = 1/3; P_after[0, 4] = 2/3; P_after[4, 3] = 1; P_after[2, 3] = 1
s_after = sim_network([1.5, 0, 0.5, 0, 0], P_after, [2.5, 1, 1, 2.5, 1], [1, np.inf, np.inf, 1, np.inf], T=40_000, rng=rng, warmup=1_000,
                      service=[None, delay2, delay2, None, delay1])
print(f"before the link: W = {s_before['W']:.3f} (theory {br['W_before']:.3f});   queue sojourns x, y = {s_before['W_i'][0]:.3f}, {s_before['W_i'][3]:.3f}")
print(f"after the link:  W = {s_after['W']:.3f} (theory {br['W_after']:.3f});   queue sojourns x, y = {s_after['W_i'][0]:.3f}, {s_after['W_i'][3]:.3f}  (both 1 at equilibrium)")
print("Route times after the link: upper", round(s_after['W_i'][0] + 2, 2), " middle", round(s_after['W_i'][0] + 1 + s_after['W_i'][3], 2),
      " lower", round(2 + s_after['W_i'][3], 2), " -> nobody gains by switching, and everybody is slower than before.")

In [ ]:
def braess_demo(mu=2.5):
    """W before / after the link as the service rate varies (lam = 1), and the planner's view at this mu."""
    mus = np.linspace(1.3, 4.0, 300)
    Wb = [braess(1.0, m)["W_before"] for m in mus]; Wa = [braess(1.0, m)["W_after"] for m in mus]
    b = braess(1.0, mu)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot(mus, Wb, label="before the link"); ax[0].plot(mus, Wa, label="after the link (equilibrium)")
    ax[0].axvline(mu, color="r", alpha=.5); ax[0].axvspan(2.0, 3.0, color="orange", alpha=.12, label="all three routes used")
    ax[0].set_ylim(2, 8); ax[0].set_xlabel("mu"); ax[0].set_ylabel("W"); ax[0].legend()
    ax[0].set_title(f"mu = {mu}: {b['W_before']:.2f} -> {b['W_after']:.2f}  ({b['regime']})")
    # planner: force a fraction f of the demand onto the middle route, the rest split evenly
    f = np.linspace(0, 2 * 1.0 - 1e-3, 300); fq = (2 * 1.0 + f) / 2            # flow through each queue
    ok = fq < mu; xq = np.where(ok, 1 / np.maximum(mu - fq, 1e-9), np.nan)
    Wavg = ((2 - f) * (xq + 2) + f * (2 * xq + 1)) / 2
    ax[1].plot(f, Wavg); ax[1].set_xlabel("flow forced onto the new link"); ax[1].set_ylabel("mean W of everybody")
    ax[1].set_ylim(2, 8); ax[1].set_title("A planner would close the link: the social optimum is f = 0")
    plt.show()

show(braess_demo, mu=FloatSlider(min=1.3, max=4.0, step=0.1, value=2.5) if HAS_W else 2.5)

**Think about it** — (i) For $\mu>2\lambda+1$ the equilibrium uses only the middle route;
compute $W$ there and find the $\mu$ beyond which the link finally *helps* (answer:
$\mu>2+\sqrt2$ for $\lambda=1$ — a link pays off only when capacity is ample). (ii) Why is the
equilibrium condition stated in *means*? (A customer cannot see the current queue
lengths here — what if she could?) (iii) Toll design: what fixed delay on the new link
restores $W_{\text{before}}$? (iv) Where else have you met "individually rational, collectively
worse"? Name one example from traffic, one from computing.

---
## Wrap-up

| question | tool | answer |
|---|---|---|
| three stations with feedback (#1) | traffic equations + $M/M/1$ nodes | $\lambda=(2.86,2.86,0.86)$, $L=23.25$, $W=11.6$; the $\rho=0.95$ station holds 20 of the 23 |
| Bay-Gull Bagels (#2) | 5-station Jackson network | $L=38.4$, $W=24$ min; Smoothies ($\rho=0.9$) is the hot spot; a deterministic cashier breaks the product form only locally |
| tandem (#3) | Burke, product form, Reich | $L=3$, $W=1.5$; output Poisson, counts independent at fixed $t$ (not across time), sojourns independent |
| feedback (#4) | two Little's laws | $L=1$: $W=1$ per customer, $0.5$ per visit; input not Poisson, law still geometric |
| split vs pooled (#5, finals) | $M/M/2$ vs $2\times M/M/1$; square-root rule | pooled $L$ is 60 % of split; $\mu^*=(2.93,1.07)$, $W^*=0.9<1.0<2.125$ |
| Braess (#6) | Wardrop equilibrium | $W$: $2.67\to3$ after a free fast link; planner would close it |

**Next:** HW5 (networks, finite capacity, $M/G/1$, and a discrete-event stress test of the
product form), then Ch.9.